# LeetCode 3442 — Maximum Difference Between Even and Odd Frequency I bằng Python

Link đề: https://leetcode.com/problems/maximum-difference-between-even-and-odd-frequency-i/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Counting (đếm tần suất)

Skeleton LeetCode:

```python
class Solution:
    def maxDifference(self, s: str) -> int:
        ...
```

Ghi chú: hai ví dụ chính lấy từ đề; mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu năm cách làm với nhau trên hàng nghìn chuỗi ngẫu nhiên đúng ràng buộc). Ví dụ nào ghi "tự dựng" là do mình thêm để minh họa — nếu cần khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm **tần suất** (số lần xuất hiện) của từng chữ cái trong `s`; chọn một chữ cái `a1` có tần suất **lẻ** và một chữ cái `a2` có tần suất **chẵn**; làm cho hiệu `freq(a1) - freq(a2)` **lớn nhất có thể**.
2. Nắm các công cụ Python: `dict` đếm tay, `collections.Counter`, kiểm tra chẵn/lẻ bằng `% 2`, `max` / `min`, giá trị "vô cực" `math.inf`, mảng đếm `26` phần tử với `ord(c) - ord('a')`, và `sorted`.
3. Đi từ cách **thử mọi cặp ký tự bằng `s.count`** → **đếm một lần bằng `Counter` rồi tách hai danh sách lẻ/chẵn** → **một lượt cập nhật `max_odd` / `min_even`** → **mảng 26 phần tử** → **sắp xếp các tần suất**.
4. Hiểu mấu chốt: hiệu `f1 - f2` lớn nhất khi `f1` **lớn nhất** trong các tần suất lẻ và `f2` **nhỏ nhất** trong các tần suất chẵn — hai việc chọn này **độc lập** nhau, nên không cần thử mọi cặp.
5. Tránh các lỗi kinh điển: chọn **lẻ nhỏ nhất − chẵn lớn nhất** (ngược), khởi tạo `min_even = 0` làm `min` luôn bằng `0`, quên rằng chữ cái **không xuất hiện** (tần suất `0`) không được tính là "chẵn", và bỏ qua việc đáp án có thể **âm**.

## 2. Đề bài & ví dụ

Cho chuỗi `s` chỉ gồm các chữ cái tiếng Anh thường.

Hãy tìm hiệu **lớn nhất** `diff = freq(a1) - freq(a2)` giữa tần suất của hai ký tự `a1` và `a2` trong chuỗi, với điều kiện:

- `a1` có tần suất **lẻ** trong `s`;
- `a2` có tần suất **chẵn** trong `s`.

Trả về hiệu lớn nhất đó.

**Ví dụ 1**

- Input: `s = "aaaaabbc"`
- Output: `3`
- Giải thích: `a` xuất hiện `5` lần (lẻ), `b` xuất hiện `2` lần (chẵn), `c` xuất hiện `1` lần (lẻ). Chọn `a1 = 'a'` (tần suất lẻ lớn nhất) và `a2 = 'b'` (tần suất chẵn duy nhất) ⇒ `5 - 2 = 3`.

**Ví dụ 2**

- Input: `s = "abcabcab"`
- Output: `1`
- Giải thích: `a` xuất hiện `3` lần (lẻ), `b` xuất hiện `3` lần (lẻ), `c` xuất hiện `2` lần (chẵn). Chọn `a1 = 'a'` (hoặc `'b'`) và `a2 = 'c'` ⇒ `3 - 2 = 1`.

**Ràng buộc:** `3 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường; `s` có **ít nhất một** ký tự tần suất lẻ và **ít nhất một** ký tự tần suất chẵn.

In [1]:
import random
import string
from collections import Counter
from math import inf

examples = [
    ("aaaaabbc", 3),
    ("abcabcab", 1),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected)

'aaaaabbc' -> kỳ vọng 3
'abcabcab' -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Tần suất (frequency) là gì? Đếm tay bằng `dict`

**Tần suất** của một ký tự là **số lần** ký tự đó xuất hiện trong chuỗi. Với `"aaaaabbc"`: `a` xuất hiện `5` lần, `b` `2` lần, `c` `1` lần.

Cách đếm tự nhiên nhất: dùng một `dict` (từ điển — cặp *khóa → giá trị*). Với mỗi ký tự `c` của chuỗi: nếu `c` đã có trong `dict` thì cộng thêm `1`, nếu chưa thì bắt đầu từ `0` rồi cộng `1`. Hàm `dict.get(c, 0)` trả về giá trị của `c`, hoặc `0` nếu `c` chưa có — rất tiện.

In [2]:
s = "aaaaabbc"
cnt = {}
for c in s:
    cnt[c] = cnt.get(c, 0) + 1      # chưa có -> 0, rồi cộng 1
print(cnt)                          # {'a': 5, 'b': 2, 'c': 1}
print(cnt["a"], cnt.get("z", 0))    # 5 0   (z không xuất hiện -> mặc định 0)

{'a': 5, 'b': 2, 'c': 1}
5 0


### 3.2. `collections.Counter`: đếm tần suất bằng một dòng

`Counter(s)` làm đúng việc ở trên: trả về một đối tượng giống `dict`, khóa là ký tự, giá trị là tần suất. Hai điểm cần nhớ:

- `Counter(s).values()` cho **các tần suất** (bỏ qua tên ký tự) — thứ ta cần trong bài này.
- Tra một ký tự **không có** trong `Counter` trả về `0` (không báo lỗi `KeyError`).

In [3]:
cnt = Counter("aaaaabbc")
print(cnt)                          # Counter({'a': 5, 'b': 2, 'c': 1})
print(list(cnt.values()))           # [5, 2, 1]
print(cnt["a"], cnt["z"])           # 5 0
print(cnt == {"a": 5, "b": 2, "c": 1})   # True (so sánh được với dict thường)

Counter({'a': 5, 'b': 2, 'c': 1})
[5, 2, 1]
5 0
True


### 3.3. Chẵn / lẻ bằng toán tử `%` (chia lấy dư)

`x % 2` là **phần dư** khi chia `x` cho `2`. Số **chẵn** có dư `0`; số **lẻ** có dư `1`. Ví dụ: `5 % 2 = 1` (lẻ), `2 % 2 = 0` (chẵn). Trong `if`, `if v % 2:` nghĩa là "nếu dư khác `0`", tức `v` **lẻ**. Đừng quên: `0` cũng là số **chẵn** (`0 % 2 == 0`) — chi tiết này sẽ quan trọng ở Cách 4.

In [4]:
for v in [0, 1, 2, 3, 4, 5]:
    print(v, "% 2 =", v % 2, "->", "lẻ" if v % 2 else "chẵn")

0 % 2 = 0 -> chẵn
1 % 2 = 1 -> lẻ
2 % 2 = 0 -> chẵn
3 % 2 = 1 -> lẻ
4 % 2 = 0 -> chẵn
5 % 2 = 1 -> lẻ


### 3.4. `max`, `min` và giá trị "vô cực" `math.inf`

`max(...)` trả về phần tử lớn nhất, `min(...)` phần tử nhỏ nhất. Khi ta **cập nhật dần** một giá trị nhỏ nhất, cần một khởi đầu mà **mọi số thực** đều nhỏ hơn: đó là `math.inf` (vô cực dương). Với giá trị lớn nhất, khởi đầu `0` đủ dùng khi mọi số đều dương.

Cẩn thận: nếu khởi tạo `min_even = 0` thì `min(0, bất_kỳ_số_dương)` luôn bằng `0` — **sai**.

In [5]:
best_min = inf
for v in [4, 2, 6]:
    best_min = min(best_min, v)
    print("v =", v, "-> min hiện tại =", best_min)

wrong = 0
for v in [4, 2, 6]:
    wrong = min(wrong, v)
print("khởi tạo 0 thì min luôn là:", wrong)     # 0  (sai)

v = 4 -> min hiện tại = 4
v = 2 -> min hiện tại = 2
v = 6 -> min hiện tại = 2
khởi tạo 0 thì min luôn là: 0


### 3.5. `s.count(c)` và `set(s)`

- `s.count(c)` đếm số lần `c` xuất hiện trong `s`; mỗi lần gọi phải quét cả chuỗi (`O(n)`).
- `set(s)` là tập hợp các ký tự **khác nhau** của `s` (không lặp, không có thứ tự). Bài này tối đa `26` chữ cái khác nhau.

In [6]:
s = "aaaaabbc"
print(s.count("a"), s.count("b"), s.count("z"))   # 5 2 0
print(sorted(set(s)))                              # ['a', 'b', 'c']
print(len(set(s)))                                 # 3

5 2 0
['a', 'b', 'c']
3


### 3.6. Mảng 26 phần tử và `ord`

Vì chỉ có chữ thường, ta có thể dùng list `26` số thay cho `dict`: `cnt[ord(c) - ord('a')] += 1`. `ord('a') = 97`, nên `ord(c) - 97` cho chỉ số `0..25` (`a -> 0`, `b -> 1`, ...). Lưu ý các chữ không xuất hiện có `cnt = 0`.

In [7]:
cnt = [0] * 26
for c in "aaaaabbc":
    cnt[ord(c) - ord("a")] += 1
print(cnt[:5])                      # [5, 2, 1, 0, 0]
print(sum(cnt))                     # 8 = len(s)
print(sum(1 for v in cnt if v == 0), "chữ cái có tần suất 0")   # 23

[5, 2, 1, 0, 0]
8
23 chữ cái có tần suất 0


### 3.7. `sorted` các tần suất

`sorted(cnt.values())` sắp xếp các tần suất tăng dần. Dễ thấy: phần tử **nhỏ nhất** ở đầu, **lớn nhất** ở cuối — gợi ý một cách làm khác ở Cách 5.

In [8]:
freqs = sorted(Counter("abcabcab").values())
print(freqs)                        # [2, 3, 3]
print(freqs[0], freqs[-1])          # 2 3

[2, 3, 3]
2 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "aaaaabbc"`. Bước 1: **đếm** từng chữ. Bước 2: **phân loại** lẻ/chẵn. Bước 3: thử các cặp (lẻ, chẵn).

| ký tự | tần suất | lẻ hay chẵn |
|---|---|---|
| a | 5 | lẻ |
| b | 2 | chẵn |
| c | 1 | lẻ |

Các cặp (lẻ, chẵn) có thể chọn:

| `a1` (lẻ) | `a2` (chẵn) | `freq(a1) - freq(a2)` |
|---|---|---|
| a (5) | b (2) | `5 - 2 = 3` |
| c (1) | b (2) | `1 - 2 = -1` |

Lớn nhất là `3`. Để hiệu `f1 - f2` **to nhất**, ta muốn số bị trừ `f1` **to nhất có thể** và số trừ `f2` **bé nhất có thể**. Vì `f1` chỉ được chọn trong nhóm lẻ, `f2` chỉ được chọn trong nhóm chẵn, nên ta lấy **lẻ lớn nhất** và **chẵn nhỏ nhất**.

In [9]:
s = "aaaaabbc"
cnt = Counter(s)
for c, f in sorted(cnt.items()):
    print(f"{c}: tần suất {f} -> {'lẻ' if f % 2 else 'chẵn'}")
odds = [f for f in cnt.values() if f % 2]
evens = [f for f in cnt.values() if f % 2 == 0]
print("các tần suất lẻ :", odds, "-> lớn nhất", max(odds))
print("các tần suất chẵn:", evens, "-> nhỏ nhất", min(evens))
print("đáp án =", max(odds) - min(evens))

a: tần suất 5 -> lẻ
b: tần suất 2 -> chẵn
c: tần suất 1 -> lẻ
các tần suất lẻ : [5, 1] -> lớn nhất 5
các tần suất chẵn: [2] -> nhỏ nhất 2
đáp án = 3


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp ký tự bằng `s.count`

**Ý tưởng.** Bám sát đề: lấy tập ký tự khác nhau `letters = set(s)`. Với **mọi cặp** `(a1, a2)`: tính `f1 = s.count(a1)`, `f2 = s.count(a2)`; nếu `f1` lẻ và `f2` chẵn thì ứng viên `f1 - f2`; giữ ứng viên lớn nhất.

Vì đề đảm bảo có ít nhất một cặp hợp lệ, biến `best` sẽ được gán ít nhất một lần (khởi tạo `None` để không nhầm với đáp án thật, vì đáp án có thể âm).

In [10]:
def max_diff_brute(s: str) -> int:
    best = None
    letters = set(s)
    for a1 in letters:
        for a2 in letters:
            f1 = s.count(a1)
            f2 = s.count(a2)
            if f1 % 2 == 1 and f2 % 2 == 0:
                diff = f1 - f2
                if best is None or diff > best:
                    best = diff
    return best


for s, expected in examples:
    print(max_diff_brute(s), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Gọi `n = len(s)` và `k` là số chữ cái khác nhau (`k <= 26`). Có `k²` cặp, mỗi cặp gọi `count` hai lần, mỗi lần `O(n)` ⇒ **Time `O(k² · n)`**, **Space `O(k)`** (tập `set`). Với `n <= 100`, `k <= 26` thì tối đa khoảng `26·26·2·100 ≈ 135 000` bước — vẫn nhanh, nhưng lãng phí.

**Nút thắt:** ta **đếm lại** cùng một chữ cái rất nhiều lần (mỗi chữ được đếm `~2k` lần). Chỉ cần đếm **một lần** cho mỗi chữ, rồi so sánh các con số đã có.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét làm bài gọn hẳn:

1. **Đếm một lần.** Một lượt duyệt `s` (bằng `Counter`) cho tần suất của mọi chữ.
2. **Không cần thử mọi cặp.** Hiệu `f1 - f2`: `f1` chọn **độc lập** trong nhóm lẻ, `f2` chọn độc lập trong nhóm chẵn. Với `f2` cố định, hiệu lớn nhất khi `f1` lớn nhất; với `f1` cố định, hiệu lớn nhất khi `f2` nhỏ nhất. Vậy đáp án là

   **`max(tần suất lẻ) - min(tần suất chẵn)`**.

Ví dụ `"abcabcab"`: tần suất `{a: 3, b: 3, c: 2}` ⇒ lẻ lớn nhất `3`, chẵn nhỏ nhất `2` ⇒ `1`.

In [11]:
cnt = Counter("abcabcab")
max_odd = max(v for v in cnt.values() if v % 2 == 1)
min_even = min(v for v in cnt.values() if v % 2 == 0)
print(cnt, "-> lẻ lớn nhất =", max_odd, ", chẵn nhỏ nhất =", min_even, ", đáp án =", max_odd - min_even)

Counter({'a': 3, 'b': 3, 'c': 2}) -> lẻ lớn nhất = 3 , chẵn nhỏ nhất = 2 , đáp án = 1


## 7. Các cách tiếp theo

### Cách 2 — `Counter` rồi tách hai danh sách lẻ / chẵn

**Giải thích từng đoạn.**

1. `cnt = Counter(s)`: tần suất mọi chữ.
2. `odds = [v for v in cnt.values() if v % 2 == 1]`: danh sách các tần suất lẻ (list comprehension: "với mỗi `v` trong ..., nếu ..., lấy `v`").
3. `evens = [v for v in cnt.values() if v % 2 == 0]`: danh sách các tần suất chẵn.
4. Trả `max(odds) - min(evens)`. Đề đảm bảo cả hai danh sách **không rỗng** nên `max`/`min` không báo lỗi.

In [12]:
def max_diff_lists(s: str) -> int:
    cnt = Counter(s)
    odds = [v for v in cnt.values() if v % 2 == 1]       # các tần suất lẻ
    evens = [v for v in cnt.values() if v % 2 == 0]      # các tần suất chẵn
    return max(odds) - min(evens)


for s, expected in examples:
    print(max_diff_lists(s), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionLists:
    def maxDifference(self, s: str) -> int:
        cnt = Counter(s)
        odds = [v for v in cnt.values() if v % 2 == 1]
        evens = [v for v in cnt.values() if v % 2 == 0]
        return max(odds) - min(evens)


print(SolutionLists().maxDifference("aaaaabbc"))      # 3

3


**Độ phức tạp Cách 2:** **Time `O(n)`** (một lượt đếm, rồi duyệt tối đa `26` tần suất), **Space `O(k)`** với `k <= 26`.

### Cách 3 — Một lượt cập nhật `max_odd` và `min_even` (không tạo danh sách phụ)

**Giải thích từng đoạn.**

1. `cnt = Counter(s)`.
2. Khởi tạo `max_odd = 0` (mọi tần suất lẻ đều `>= 1 > 0`, nên `0` an toàn làm điểm bắt đầu cho "lớn nhất") và `min_even = inf` (mọi tần suất chẵn đều nhỏ hơn `inf`).
3. Với mỗi tần suất `v`: nếu lẻ thì `max_odd = max(max_odd, v)`, ngược lại `min_even = min(min_even, v)`.
4. Trả `max_odd - min_even`.

Đây là cách ngắn gọn, ít bộ nhớ phụ và là bản mình khuyên **nộp**.

In [14]:
def max_diff_one_pass(s: str) -> int:
    cnt = Counter(s)
    max_odd = 0              # tần suất lẻ lớn nhất (mọi tần suất lẻ >= 1)
    min_even = inf           # tần suất chẵn nhỏ nhất
    for v in cnt.values():
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


for s, expected in examples:
    print(max_diff_one_pass(s), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


In [15]:
# Code nộp LeetCode cho Cách 3
class SolutionOnePass:
    def maxDifference(self, s: str) -> int:
        cnt = Counter(s)
        max_odd, min_even = 0, inf
        for v in cnt.values():
            if v % 2 == 1:
                max_odd = max(max_odd, v)
            else:
                min_even = min(min_even, v)
        return max_odd - min_even


print(SolutionOnePass().maxDifference("abcabcab"))    # 1

1


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(k)`** với `k <= 26` (tức `O(1)` vì bảng chữ cái cố định).

### Cách 4 — Mảng 26 phần tử thay cho `Counter`

**Giải thích từng đoạn.**

1. `cnt = [0] * 26`; mỗi ký tự `c`: `cnt[ord(c) - ord('a')] += 1`.
2. Duyệt `26` ô. **Chú ý quan trọng:** chữ cái **không xuất hiện** có `cnt = 0`, mà `0 % 2 == 0` trông như "chẵn" — nhưng đề nói chữ phải **có mặt** trong `s`. Do đó **bỏ qua** `v == 0`.
3. Phần còn lại giống Cách 3.

In [16]:
def max_diff_array(s: str) -> int:
    cnt = [0] * 26
    for c in s:
        cnt[ord(c) - ord("a")] += 1
    max_odd, min_even = 0, inf
    for v in cnt:
        if v == 0:                       # chữ cái không xuất hiện: bỏ qua
            continue
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


for s, expected in examples:
    print(max_diff_array(s), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n + 26)` = `O(n)`**, **Space `O(26) = O(1)`**.

### Cách 5 — Sắp xếp các tần suất rồi lấy hai đầu

**Giải thích.** Sắp xếp tăng dần các tần suất. Tần suất **chẵn nhỏ nhất** là phần tử chẵn **đầu tiên** khi đọc từ trái; tần suất **lẻ lớn nhất** là phần tử lẻ **đầu tiên** khi đọc từ phải. `next(generator)` lấy phần tử đầu tiên thỏa điều kiện. Cách này tốn thêm bước sắp xếp (`O(k log k)`, `k <= 26`) nhưng minh họa rõ "hai đầu của danh sách đã sắp".

In [17]:
def max_diff_sorted(s: str) -> int:
    freqs = sorted(Counter(s).values())                        # tăng dần
    min_even = next(v for v in freqs if v % 2 == 0)            # chẵn đầu tiên từ trái
    max_odd = next(v for v in reversed(freqs) if v % 2 == 1)   # lẻ đầu tiên từ phải
    return max_odd - min_even


for s, expected in examples:
    print(max_diff_sorted(s), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )


**Độ phức tạp Cách 5:** **Time `O(n + k log k)`**, **Space `O(k)`** — với `k <= 26` thực tế vẫn như `O(n)`.

## 8. TRACE từng bước

Ta trace Cách 3 (một lượt cập nhật) trên `s = "aaaaabbc"` và `s = "abcabcab"`, cho thấy `max_odd` và `min_even` thay đổi thế nào qua từng tần suất.

In [18]:
def trace_one_pass(s: str) -> int:
    cnt = Counter(s)
    print(f"s = {s!r}, tần suất = {dict(cnt)}")
    max_odd, min_even = 0, inf
    for c, v in cnt.items():
        if v % 2 == 1:
            old = max_odd
            max_odd = max(max_odd, v)
            print(f"  {c!r}: tần suất {v} là LẺ  -> max_odd: {old} -> {max_odd}")
        else:
            old = min_even
            min_even = min(min_even, v)
            print(f"  {c!r}: tần suất {v} là CHẴN -> min_even: {old} -> {min_even}")
    print(f"  kết quả = {max_odd} - {min_even} = {max_odd - min_even}")
    return max_odd - min_even


assert trace_one_pass("aaaaabbc") == 3
assert trace_one_pass("abcabcab") == 1

s = 'aaaaabbc', tần suất = {'a': 5, 'b': 2, 'c': 1}
  'a': tần suất 5 là LẺ  -> max_odd: 0 -> 5
  'b': tần suất 2 là CHẴN -> min_even: inf -> 2
  'c': tần suất 1 là LẺ  -> max_odd: 5 -> 5
  kết quả = 5 - 2 = 3
s = 'abcabcab', tần suất = {'a': 3, 'b': 3, 'c': 2}
  'a': tần suất 3 là LẺ  -> max_odd: 0 -> 3
  'b': tần suất 3 là LẺ  -> max_odd: 3 -> 3
  'c': tần suất 2 là CHẴN -> min_even: inf -> 2
  kết quả = 3 - 2 = 1


In [19]:
def trace_brute(s: str) -> int:
    best = None
    letters = sorted(set(s))
    print(f"s = {s!r}, các chữ = {letters}")
    for a1 in letters:
        for a2 in letters:
            f1, f2 = s.count(a1), s.count(a2)
            ok = f1 % 2 == 1 and f2 % 2 == 0
            if ok:
                if best is None or f1 - f2 > best:
                    best = f1 - f2
                print(f"  (a1={a1!r}, a2={a2!r}): f1={f1} lẻ, f2={f2} chẵn -> {f1 - f2}, best = {best}")
    return best


assert trace_brute("abcabcab") == 1

s = 'abcabcab', các chữ = ['a', 'b', 'c']
  (a1='a', a2='c'): f1=3 lẻ, f2=2 chẵn -> 1, best = 1
  (a1='b', a2='c'): f1=3 lẻ, f2=2 chẵn -> 1, best = 1


Bảng trace Cách 3 với `s = "aaaaabbc"` (tần suất `a: 5, b: 2, c: 1`):

| bước | ký tự | tần suất `v` | lẻ/chẵn | `max_odd` | `min_even` |
|---|---|---|---|---|---|
| bắt đầu | — | — | — | 0 | inf |
| 1 | a | 5 | lẻ | 5 | inf |
| 2 | b | 2 | chẵn | 5 | 2 |
| 3 | c | 1 | lẻ | 5 (vì `max(5, 1) = 5`) | 2 |

Kết quả `5 - 2 = 3`. Bảng trace với `s = "abcabcab"` (`a: 3, b: 3, c: 2`): `max_odd` đi `0 → 3 → 3`, `min_even` đi `inf → 2`; kết quả `3 - 2 = 1`.

## 9. Code nộp LeetCode

Mình nộp Cách 3 (một lượt, ít bộ nhớ). Trên LeetCode, `Counter` và `inf` thường đã được nạp sẵn trong môi trường; để chạy ở máy bạn, ghi rõ dòng `import`.

In [20]:
from collections import Counter
from math import inf


class Solution:
    def maxDifference(self, s: str) -> int:
        cnt = Counter(s)
        max_odd = 0              # tần suất lẻ lớn nhất
        min_even = inf           # tần suất chẵn nhỏ nhất
        for v in cnt.values():
            if v % 2 == 1:
                max_odd = max(max_odd, v)
            else:
                min_even = min(min_even, v)
        return max_odd - min_even


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.maxDifference(s), "(kỳ vọng", expected, ")")

'aaaaabbc' -> 3 (kỳ vọng 3 )
'abcabcab' -> 1 (kỳ vọng 1 )


## 10. Vì sao thuật toán đúng

- **Đếm đủ và đúng:** `Counter(s)` cho tần suất chính xác của mọi ký tự có mặt (mỗi ký tự có mặt có tần suất `>= 1`), nên ta không nhầm ký tự vắng mặt thành "chẵn".
- **Hai lựa chọn độc lập:** tập ứng viên cho `a1` (tần suất lẻ) và tập ứng viên cho `a2` (tần suất chẵn) **không giao nhau** (một tần suất không thể vừa chẵn vừa lẻ), và có thể chọn bất kỳ phần tử nào ở mỗi tập (hai chữ khác nhau vì tần suất một chẵn một lẻ). Do đó mọi cặp (lẻ, chẵn) đều hợp lệ.
- **Cực đại hóa:** `f1 - f2` tăng theo `f1` và giảm theo `f2`, nên cực đại tại `f1 = max(lẻ)` và `f2 = min(chẵn)`. Nếu có cặp nào tốt hơn thì `f1' <= max_odd` và `f2' >= min_even` ⇒ `f1' - f2' <= max_odd - min_even`.
- **Khởi tạo an toàn:** `max_odd = 0` không bao giờ thắng một tần suất lẻ (`>= 1`); `min_even = inf` luôn bị thay bởi tần suất chẵn đầu tiên; đề đảm bảo cả hai nhóm không rỗng nên kết quả là số nguyên hữu hạn.
- **Đáp án có thể âm:** ví dụ `"abb"` có `a: 1` (lẻ), `b: 2` (chẵn) ⇒ `1 - 2 = -1`. Ta **không** dùng `abs` hay ép `>= 0`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)` và `k <= 26` là số chữ cái khác nhau.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cặp bằng `s.count` | `O(k² · n)` | `O(k)` |
| 2. `Counter` + hai danh sách | `O(n)` | `O(k)` |
| 3. `Counter` + `max_odd`/`min_even` | `O(n)` | `O(k)` |
| 4. Mảng 26 phần tử | `O(n)` | `O(1)` |
| 5. Sắp xếp các tần suất | `O(n + k log k)` | `O(k)` |

Vì `k <= 26` là hằng số, mọi cách (trừ Cách 1) thực chất là `O(n)` thời gian và `O(1)` bộ nhớ. Với `n <= 100`, mọi cách đều chạy tức thì.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng bằng `assert`.

In [21]:
sol = Solution()
impls = {
    "brute": max_diff_brute,
    "lists": max_diff_lists,
    "one_pass": max_diff_one_pass,
    "array": max_diff_array,
    "sorted": max_diff_sorted,
    "leetcode": sol.maxDifference,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2


### 12.1. Các trường hợp biên

Chú ý các ca **đáp án âm**, **độ dài nhỏ nhất `3`** và nhiều chữ cùng tần suất. (Các chuỗi tự dựng bên dưới đã được kiểm tra thủ công đúng ràng buộc: có ít nhất một tần suất lẻ và một chẵn.)

In [22]:
check_all("abb", -1, "độ dài 3, a=1 lẻ, b=2 chẵn -> 1 - 2 = -1")
check_all("aab", -1, "a=2 chẵn, b=1 lẻ -> 1 - 2 = -1")
check_all("aaabb", 3 - 2, "a=3 lẻ, b=2 chẵn")
check_all("aabbb", 3 - 2, "a=2 chẵn, b=3 lẻ")
check_all("mississippi", -1, "m=1, i=4, s=4, p=2 -> 1 - 2")
check_all("aabbbcccc", 3 - 2, "a=2, b=3, c=4 -> lẻ lớn nhất 3, chẵn nhỏ nhất 2")
check_all("a" * 97 + "bb", 97 - 2, "a=97 lẻ, b=2 chẵn, độ dài 99")
check_all("abbccc", 3 - 2, "a=1, b=2, c=3")
check_all("abbcccddddd", 5 - 2, "a=1, b=2, c=3, d=5 -> 5 - 2")
check_all("aabbccd", 1 - 2, "a=b=c=2 chẵn, d=1 lẻ -> 1 - 2")

PASS độ dài 3, a=1 lẻ, b=2 chẵn -> 1 - 2 = -1
PASS a=2 chẵn, b=1 lẻ -> 1 - 2 = -1
PASS a=3 lẻ, b=2 chẵn
PASS a=2 chẵn, b=3 lẻ
PASS m=1, i=4, s=4, p=2 -> 1 - 2
PASS a=2, b=3, c=4 -> lẻ lớn nhất 3, chẵn nhỏ nhất 2
PASS a=97 lẻ, b=2 chẵn, độ dài 99
PASS a=1, b=2, c=3
PASS a=1, b=2, c=3, d=5 -> 5 - 2
PASS a=b=c=2 chẵn, d=1 lẻ -> 1 - 2


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "trông có vẻ đúng" nhưng sai: (a) lấy **lẻ nhỏ nhất − chẵn lớn nhất** (ngược); (b) bỏ qua tính chẵn/lẻ, lấy **max − min** của mọi tần suất; (c) khởi tạo `min_even = 0`.

In [23]:
def wrong_reversed(s):
    cnt = Counter(s)
    odds = [v for v in cnt.values() if v % 2 == 1]
    evens = [v for v in cnt.values() if v % 2 == 0]
    return min(odds) - max(evens)                # SAI: ngược (nhỏ nhất - lớn nhất)


def wrong_ignore_parity(s):
    cnt = Counter(s)
    return max(cnt.values()) - min(cnt.values())    # SAI: không xét chẵn/lẻ


def wrong_init_zero(s):
    cnt = Counter(s)
    max_odd, min_even = 0, 0                    # SAI: min_even = 0
    for v in cnt.values():
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


for fn in (wrong_reversed, wrong_ignore_parity, wrong_init_zero):
    print(fn.__name__, fn("aaaaabbc"), "!=", max_diff_one_pass("aaaaabbc"))
assert wrong_reversed("aaaaabbc") != 3
assert wrong_ignore_parity("abcabcab") != 1 or wrong_ignore_parity("aaaaabbc") != 3
assert wrong_init_zero("aaaaabbc") != 3

wrong_reversed -1 != 3
wrong_ignore_parity 4 != 3
wrong_init_zero 5 != 3


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi ngẫu nhiên **đúng ràng buộc**: độ dài `3..100`, chữ thường, có ít nhất một tần suất lẻ và một tần suất chẵn (thử lại nếu chưa thỏa — kỹ thuật *rejection sampling*). Với mọi chuỗi, cả sáu hàm phải cho cùng kết quả với Cách 1 (vét cạn đúng theo định nghĩa đề).

In [24]:
def valid(s):
    freqs = Counter(s).values()
    return (3 <= len(s) <= 100
            and all("a" <= c <= "z" for c in s)
            and any(v % 2 == 1 for v in freqs)
            and any(v % 2 == 0 for v in freqs))


rng = random.Random(3442)
tested = 0
while tested < 3000:
    n = rng.randint(3, 100)
    alphabet_size = rng.choice([2, 3, 5, 10, 26])
    letters = rng.sample(string.ascii_lowercase, alphabet_size)
    s = "".join(rng.choice(letters) for _ in range(n))
    if not valid(s):
        continue
    ref = max_diff_brute(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s, fn(s), ref)
    tested += 1
print("PASS", tested, "chuỗi ngẫu nhiên đúng ràng buộc")

PASS 3000 chuỗi ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lấy ngược — chẵn lớn nhất và lẻ nhỏ nhất

In [25]:
def sai_nguoc(s):
    cnt = Counter(s)
    odds = [v for v in cnt.values() if v % 2 == 1]
    evens = [v for v in cnt.values() if v % 2 == 0]
    return min(odds) - max(evens)      # SAI: muốn hiệu lớn nhất thì lẻ phải lớn, chẵn phải nhỏ


print("sai:", sai_nguoc("aaaaabbc"), "| đúng:", Solution().maxDifference("aaaaabbc"))

sai: -1 | đúng: 3


**Vì sao sai:** hiệu `f1 - f2` lớn nhất khi số bị trừ `f1` lớn và số trừ `f2` nhỏ. Lấy ngược sẽ cho hiệu **nhỏ nhất**. Ở ví dụ 1 ra `1 - 2 = -1` thay vì `3`. **Cách sửa:** `max(lẻ) - min(chẵn)`.

### Lỗi 2: Khởi tạo `min_even = 0`

In [26]:
def sai_khoi_tao(s):
    cnt = Counter(s)
    max_odd, min_even = 0, 0               # SAI: 0 nhỏ hơn mọi tần suất, min_even không bao giờ đổi
    for v in cnt.values():
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


print("sai:", sai_khoi_tao("aaaaabbc"), "| đúng:", Solution().maxDifference("aaaaabbc"))

sai: 5 | đúng: 3


**Vì sao sai:** `min(0, 2) = 0`; giá trị khởi đầu `0` thắng mọi tần suất dương, nên `min_even` kẹt ở `0` và kết quả thành `5 - 0 = 5`. **Cách sửa:** khởi tạo `min_even = inf` (hoặc `len(s) + 1`); với `max_odd` thì `0` là an toàn vì mọi tần suất lẻ `>= 1`.

### Lỗi 3: Dùng mảng 26 mà quên bỏ qua tần suất `0`

In [27]:
def sai_quen_bo_qua_0(s):
    cnt = [0] * 26
    for c in s:
        cnt[ord(c) - 97] += 1
    max_odd, min_even = 0, inf
    for v in cnt:                          # SAI: các chữ vắng mặt có v = 0, bị tính là chẵn
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


print("sai:", sai_quen_bo_qua_0("aaaaabbc"), "| đúng:", Solution().maxDifference("aaaaabbc"))

sai: 5 | đúng: 3


**Vì sao sai:** `0 % 2 == 0` nên các chữ cái **không xuất hiện** bị coi là "tần suất chẵn bằng `0`", kéo `min_even` về `0`. Đề nói `a2` phải là ký tự **trong chuỗi** có tần suất chẵn. **Cách sửa:** `if v == 0: continue` (Cách 4) hoặc dùng `Counter` (chỉ chứa chữ có mặt).

### Lỗi 4: Nghĩ đáp án luôn dương và dùng `abs`, hoặc `max(0, ...)`

In [28]:
def sai_abs(s):
    cnt = Counter(s)
    max_odd, min_even = 0, inf
    for v in cnt.values():
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return abs(max_odd - min_even)         # SAI: đề cần hiệu có dấu freq(a1) - freq(a2)


print("sai:", sai_abs("abb"), "| đúng:", Solution().maxDifference("abb"))

sai: 1 | đúng: -1


**Vì sao sai:** với `"abb"`, `a1` phải là ký tự lẻ (`a`, tần suất `1`) và `a2` chẵn (`b`, tần suất `2`) nên hiệu là `1 - 2 = -1`. Đề **không** cho đổi vai trò hai ký tự, nên không được lấy trị tuyệt đối. **Cách sửa:** trả đúng `max_odd - min_even`, kể cả khi âm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề có chữ "**tần suất**" / "số lần xuất hiện" ⇒ nghĩ ngay tới `Counter` (hoặc mảng `26`).
- "Chọn **hai** phần tử, một thỏa điều kiện A, một thỏa điều kiện B, tối ưu hiệu/tổng" ⇒ thường **tách thành hai việc độc lập**: tìm cực trị của nhóm A và nhóm B.
- Muốn **hiệu lớn nhất**: số bị trừ **lớn nhất**, số trừ **nhỏ nhất**. Muốn **hiệu nhỏ nhất**: ngược lại.
- Khởi tạo cực trị: `max` → bắt đầu `0` (khi mọi giá trị dương) hoặc `-inf`; `min` → bắt đầu `inf`.
- Chẵn/lẻ: `v % 2 == 1` (lẻ), `v % 2 == 0` (chẵn); nhớ `0` là chẵn.
- Khi dùng mảng `26`, hãy tự hỏi: "chữ vắng mặt (`0`) có làm hỏng điều kiện không?".
- Kiểm tra nhanh trước khi nộp: tự đếm tay tần suất của hai ví dụ trong đề.

## 15. Biến thể của bài

1. **Ngược lại:** hiệu lớn nhất `freq(chẵn) - freq(lẻ)` ⇒ `max(chẵn) - min(lẻ)`.
2. **Hiệu nhỏ nhất** `freq(lẻ) - freq(chẵn)` ⇒ `min(lẻ) - max(chẵn)`.
3. **Liệt kê mọi hiệu** hợp lệ (tất cả cặp lẻ–chẵn), hoặc **đếm** số cặp hợp lệ.
4. **Bài II (Hard, LeetCode 3445):** tìm hiệu lớn nhất trên **chuỗi con** có độ dài `>= k` — cần cửa sổ trượt và kỹ thuật trạng thái chẵn/lẻ.
5. Bài cùng họ đếm tần suất: **First Unique Character in a String (387)**, **Sort Characters By Frequency (451)**, **Determine if Two Strings Are Close (1657)**.

In [29]:
def max_even_minus_odd(s):
    cnt = Counter(s)
    odds = [v for v in cnt.values() if v % 2 == 1]
    evens = [v for v in cnt.values() if v % 2 == 0]
    return max(evens) - min(odds)


def min_odd_minus_even(s):
    cnt = Counter(s)
    odds = [v for v in cnt.values() if v % 2 == 1]
    evens = [v for v in cnt.values() if v % 2 == 0]
    return min(odds) - max(evens)


def all_valid_diffs(s):
    cnt = Counter(s)
    return sorted((fo - fe for fo in cnt.values() if fo % 2 == 1 for fe in cnt.values() if fe % 2 == 0), reverse=True)


print(max_even_minus_odd("aaaaabbc"))      # chẵn lớn nhất 2, lẻ nhỏ nhất 1 -> 1
print(min_odd_minus_even("aaaaabbc"))      # lẻ nhỏ nhất 1, chẵn lớn nhất 2 -> -1
print(all_valid_diffs("aaaaabbc"))         # [3, -1]
assert max_even_minus_odd("aaaaabbc") == 1
assert min_odd_minus_even("aaaaabbc") == -1
assert all_valid_diffs("aaaaabbc") == [3, -1]
assert all_valid_diffs("abcabcab")[0] == Solution().maxDifference("abcabcab")

1
-1
[3, -1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cặp + `count` | làm đúng theo đề, đếm lại mỗi lần | `O(k² · n)` | `O(k)` | để hiểu đề, đối chiếu |
| 2. `Counter` + hai danh sách | tách tần suất lẻ / chẵn rồi `max`/`min` | `O(n)` | `O(k)` | dễ đọc nhất |
| 3. `Counter` + cập nhật dần | `max_odd`, `min_even` trong một lượt | `O(n)` | `O(k)` | **nên nộp** |
| 4. Mảng 26 | `ord(c) - ord('a')`, bỏ qua `0` | `O(n)` | `O(1)` | chỉ có chữ thường, gọn bộ nhớ |
| 5. Sắp xếp tần suất | hai đầu của danh sách đã sắp | `O(n + k log k)` | `O(k)` | minh họa ý tưởng cực trị |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter("aab")["a"]`, `7 % 2`, `max([3, 5, 1])`, `min([4, 2, 6])`, `Solution().maxDifference("abb")`.

**Bài 2 (tính tay).** Với `s = "mississippi"`: lập bảng tần suất của từng chữ, phân loại lẻ/chẵn, rồi tính đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_difference_todo(s)` theo Cách 3.

In [30]:
def max_difference_todo(s):
    # TODO 1: đếm tần suất bằng Counter
    # TODO 2: max_odd = 0, min_even = inf
    # TODO 3: duyệt các tần suất v: lẻ -> cập nhật max_odd, chẵn -> cập nhật min_even
    # TODO 4: trả về max_odd - min_even
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_difference_array2(s)` theo Cách 4 **không dùng `Counter` và không dùng `ord`**: dùng `string.ascii_lowercase` để duyệt từng chữ và `s.count(ch)` để lấy tần suất (bỏ qua chữ có tần suất `0`).

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_valid_pairs(s)` đếm **số cặp ký tự (a1, a2)** hợp lệ (a1 tần suất lẻ, a2 tần suất chẵn). (b) Viết `is_valid_input(s)` kiểm tra `s` thỏa ràng buộc của đề (độ dài `3..100`, chữ thường, có ít nhất một tần suất lẻ và một chẵn).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2`, `1`, `5`, `2`, `-1`. Bài 3: `Counter(s)`, hai biến `max_odd`, `min_even`, một vòng `for v in cnt.values()`.
- **Mức 2 (vừa):** Bài 2: `m = 1` (lẻ), `i = 4` (chẵn), `s = 4` (chẵn), `p = 2` (chẵn) ⇒ lẻ lớn nhất `1`, chẵn nhỏ nhất `2` ⇒ `1 - 2 = -1`. Bài 4: `for ch in string.ascii_lowercase: v = s.count(ch)`.
- **Mức 3 (sâu):** Bài 5(a): đếm số tần suất lẻ `x` và số tần suất chẵn `y` (tính theo **ký tự khác nhau**), đáp án `x * y`. 5(b): dùng `Counter` và `any`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert Counter("aab")["a"] == 2
assert 7 % 2 == 1
assert max([3, 5, 1]) == 5
assert min([4, 2, 6]) == 2
assert Solution().maxDifference("abb") == -1

# Bài 2
cnt_ms = Counter("mississippi")
assert dict(cnt_ms) == {"m": 1, "i": 4, "s": 4, "p": 2}
assert Solution().maxDifference("mississippi") == 1 - 2


# Bài 3
def max_difference_todo_ans(s):
    cnt = Counter(s)
    max_odd, min_even = 0, inf
    for v in cnt.values():
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


for s, expected in examples:
    assert max_difference_todo_ans(s) == expected


# Bài 4
def max_difference_array2(s):
    max_odd, min_even = 0, inf
    for ch in string.ascii_lowercase:
        v = s.count(ch)
        if v == 0:
            continue
        if v % 2 == 1:
            max_odd = max(max_odd, v)
        else:
            min_even = min(min_even, v)
    return max_odd - min_even


for s, expected in examples:
    assert max_difference_array2(s) == expected
assert max_difference_array2("mississippi") == -1


# Bài 5(a)
def count_valid_pairs(s):
    cnt = Counter(s)
    x = sum(1 for v in cnt.values() if v % 2 == 1)
    y = sum(1 for v in cnt.values() if v % 2 == 0)
    return x * y


assert count_valid_pairs("aaaaabbc") == 2 * 1      # lẻ: a, c; chẵn: b
assert count_valid_pairs("abcabcab") == 2 * 1      # lẻ: a, b; chẵn: c
assert count_valid_pairs("mississippi") == 1 * 3


# Bài 5(b)
def is_valid_input(s):
    cnt = Counter(s)
    return (3 <= len(s) <= 100
            and all("a" <= c <= "z" for c in s)
            and any(v % 2 == 1 for v in cnt.values())
            and any(v % 2 == 0 for v in cnt.values()))


assert is_valid_input("aaaaabbc")
assert is_valid_input("abcabcab")
assert not is_valid_input("aabb")          # không có tần suất lẻ
assert not is_valid_input("abc")           # không có tần suất chẵn
assert not is_valid_input("ab")            # quá ngắn
assert not is_valid_input("aAb")           # có chữ hoa
print("Tất cả đáp án bài tập đều đúng")

Tất cả đáp án bài tập đều đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tần suất lẻ lớn nhất trừ tần suất chẵn nhỏ nhất)

```python
from collections import Counter
from math import inf


class Solution:
    def maxDifference(self, s: str) -> int:
        cnt = Counter(s)
        max_odd, min_even = 0, inf
        for v in cnt.values():
            if v % 2 == 1:
                max_odd = max(max_odd, v)
            else:
                min_even = min(min_even, v)
        return max_odd - min_even
```

**Checklist**

- [ ] Lẻ thì lấy **lớn nhất**, chẵn thì lấy **nhỏ nhất** (không đảo)?
- [ ] `min_even` khởi tạo `inf` (không phải `0`)?
- [ ] Chỉ xét chữ **có mặt** trong `s` (không tính tần suất `0` là chẵn)?
- [ ] Không dùng `abs`: đáp án có thể âm?
- [ ] Đã thử `"aaaaabbc"` (`3`), `"abcabcab"` (`1`), `"abb"` (`-1`)?
- [ ] Skeleton đúng `class Solution` / `def maxDifference(self, s: str) -> int`?